In [ ]:
# arreglado despues del codigo anotado en la siguiente celda 26/09/2025

import tkinter as tk
from tkinter import ttk
from tkinterdnd2 import DND_FILES, TkinterDnD
import pandas as pd
import os

did_time = 7200  # 2 horas: 7200    ;   4 horas: 14400

def merge_overlapping_events(df, start_col='Event Start Time', dur_col='Duration Increase'):
    """
    Une eventos solapados (IBI < 0). Asume df ORDENADO por start_col.
    Devuelve un nuevo DataFrame con columnas:
      - Event Start Time
      - Duration Increase
      - InterBout Interval
      - Bout
    """
    if df.empty:
        out = df[[start_col, dur_col]].copy()
        out['Bout'] = []
        out['InterBout Interval'] = []
        return out

    # Construimos lista de (start, end)
    events = []
    for _, row in df.iterrows():
        start = float(row[start_col])
        dur = float(row[dur_col])
        end = start + dur
        events.append([start, end])

    # Merge de solapados: si start_i < end_prev => IBI < 0 => fusionar
    merged = []
    curr_start, curr_end = events[0]
    for i in range(1, len(events)):
        s, e = events[i]
        if s < curr_end:  # solape: unir
            curr_end = max(curr_end, e)
        else:
            merged.append([curr_start, curr_end])
            curr_start, curr_end = s, e
    merged.append([curr_start, curr_end])

    # Construir DF con merged
    m_df = pd.DataFrame(merged, columns=[start_col, 'Event End Time'])
    m_df['Duration Increase'] = m_df['Event End Time'] - m_df[start_col]

    # Renumerar bouts
    m_df['Bout'] = range(1, len(m_df) + 1)

    # Recalcular IBI: start_i - end_{i-1}; primera es NaN
    m_df['InterBout Interval'] = m_df[start_col].diff() - m_df['Duration Increase'].shift()

    # Dejar columnas finales consistentes
    return m_df[[start_col, 'Duration Increase', 'InterBout Interval', 'Bout']]

class App:
    def __init__(self, root):
        self.root = root
        self.setup_ui()
        self.files = []  # Lista para almacenar las rutas de los archivos a procesar

    def setup_ui(self):
        self.root.title('Arrastrar y Soltar Archivos')
        self.label = tk.Label(self.root, text='Arrastra todos los archivos y carpetas aquí', bg='lightgrey', width=50, height=10)
        self.label.pack(fill=tk.BOTH, expand=True)
        self.label.drop_target_register(DND_FILES)
        self.label.dnd_bind('<<Drop>>', self.drop)
        self.submit_button = ttk.Button(self.root, text='Enviar', command=self.process_files)
        self.submit_button.pack(side=tk.BOTTOM, pady=5)

    def drop(self, event):
        # Usa el intérprete Tcl de la ventana para dividir correctamente
        # rutas con espacios y llaves {…}
        try:
            items = self.root.tk.splitlist(event.data)
        except Exception:
            # Fallback muy básico si algo raro pasa
            items = str(event.data).split()

        for item in items:
            path = item.strip()
            # Ya no hace falta quitar {} manualmente: splitlist lo maneja
            if os.path.isdir(path):
                for rootdir, dirs, files in os.walk(path):
                    for file in files:
                        if file.lower().endswith('.csv'):
                            self.files.append(os.path.join(rootdir, file))
            else:
                if path.lower().endswith('.csv'):
                    self.files.append(path)

        # Desduplicar preservando orden (por si arrastras 2 veces lo mismo)
        self.files = list(dict.fromkeys(self.files))

        self.label.config(text=f"{len(self.files)} archivos listos para procesar")



    def process_files(self):
        for file in self.files:
            print(f"Procesando archivo: {file}")
            self.process_and_save_data(file)
        self.files = []  # Limpiar la lista de archivos después de procesar
        self.label.config(text='Arrastra todos los archivos y carpetas aquí')

    def calculate_segments(self, data, segments_seconds, label_suffix):
        segments_minutes = [(start / 60, end / 60) for start, end in zip(segments_seconds[:-1], segments_seconds[1:])]
        bouts_sum = []
        time_per_bout_sum = []
        interbout_interval_segments = []
        event_frequency_per_segment = []
        time_segment_labels = []

        for (start_minute, end_minute), end_second in zip(segments_minutes, segments_seconds[1:]):
            mask = (data['Event Start Time'] >= start_minute * 60) & (data['Event Start Time'] < end_second)
            segment_bouts = data.loc[mask, 'Bout'].count()
            bouts_sum.append(segment_bouts)
            time_per_bout_sum.append(data.loc[mask, 'Duration Increase'].sum() / 60)
            interbout_interval_sum = data.loc[mask, 'InterBout Interval'].sum(skipna=True) /60
            interbout_interval_segments.append(interbout_interval_sum)
            segment_duration_minutes = end_minute - start_minute
            frequency = segment_bouts / segment_duration_minutes if segment_duration_minutes > 0 else 0
            event_frequency_per_segment.append(frequency)
            time_segment_labels.append(f"{int(start_minute)} - {int(end_minute)} {label_suffix}")

        return pd.DataFrame({
            f'Time Segment {label_suffix}': time_segment_labels,
            f'Bouts {label_suffix}': bouts_sum,
            f'Time per Bout {label_suffix}': time_per_bout_sum,
            f'InterBout Interval {label_suffix}': interbout_interval_segments,
            f'Event Frequency per Segment {label_suffix}': event_frequency_per_segment
        })

    def process_and_save_data(self, filepath):
        # Cargar los datos
        data = pd.read_csv(filepath, encoding='ISO-8859-1')
        data.columns = data.columns.str.strip()

        # Calcular el incremento en 'LeftDuration'
        data['Duration Increase'] = data['LeftDuration'].diff().fillna(data['LeftDuration'])
        
        # Filtrar los datos para los cambios en 'LeftDuration'
        data = data[data['Duration Increase'] >= 0.5].copy()

        # ===================== CAMBIO A PARTIR DE AQUÍ =====================
        # Tiempo de inicio del evento
        data['Event Start Time'] = data['ElapsedSecs'] - data['Duration Increase']

        # Ordenar por inicio de evento
        data = data.sort_values('Event Start Time').reset_index(drop=True)

        # FUSIONAR eventos con IBI < 0 (solapados) => 1 solo evento
        merged = merge_overlapping_events(
            data,
            start_col='Event Start Time',
            dur_col='Duration Increase'
        )

        # Calcular totales sobre los eventos fusionados
        total_bouts = len(merged)                       # Total de eventos (bouts) ya fusionados
        total_bout_time = merged['Duration Increase'].sum()  # Suma total de duraciones

        # Frecuencia global por minuto usando la ventana fija (tu variable global did_time)
        # (si quieres, puedes inferirla del archivo en vez de usar did_time fijo)
        total_window = did_time
        merged.insert(0, 'Total Bout', pd.NA)
        merged.insert(1, 'Total Bout Time', pd.NA)
        merged.insert(2, 'Event Frequency per minute', pd.NA)  # mantengo tu nombre original

        if len(merged) > 0 and total_window and total_window > 0:
            merged.at[0, 'Total Bout'] = total_bouts
            merged.at[0, 'Total Bout Time'] = total_bout_time
            merged.at[0, 'Event Frequency per minute'] = total_bouts / (total_window / 60)

        # Definir los segmentos de tiempo para agregados
        segments_seconds_10min = list(range(0, total_window + 1, 600))
        segments_seconds_15min = list(range(0, total_window + 1, 900))
        segments_seconds_30min = list(range(0, total_window + 1, 1800))
        segments_seconds_60min = list(range(0, total_window + 1, 3600))

        # Calcular resultados por segmentos usando la tabla fusionada
        results_10min = self.calculate_segments(merged, segments_seconds_10min, '10min')
        results_15min = self.calculate_segments(merged, segments_seconds_15min, '15min')
        results_30min = self.calculate_segments(merged, segments_seconds_30min, '30min')
        results_60min = self.calculate_segments(merged, segments_seconds_60min, '60min')

        # Concatenar todos los DataFrames de resultados con la tabla fusionada
        base_cols = [
            'Event Start Time',
            'Duration Increase',
            'InterBout Interval',
            'Total Bout',
            'Total Bout Time',
            'Event Frequency per minute',
            'Bout'
        ]
        final_data = pd.concat(
            [
                merged[base_cols].reset_index(drop=True),
                results_10min, results_15min, results_30min, results_60min
            ],
            axis=1
        )
        # ===================== FIN DEL CAMBIO =====================

        # Guardar el DataFrame final
        directory = os.path.dirname(filepath)
        file_name = os.path.splitext(os.path.basename(filepath))[0]
        final_file_path = os.path.join(directory, f"{file_name}_procesado.xlsx")
        final_data.to_excel(final_file_path, index=False)
        print(f"Archivo guardado en: {final_file_path}")



if __name__ == "__main__":
    root = TkinterDnD.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
%pip install tkinterdnd2

In [ ]:
#Este funciona 26/09/2025

import tkinter as tk
from tkinter import ttk
from tkinterdnd2 import DND_FILES, TkinterDnD
import pandas as pd
import os

# Ventana total fija (en segundos) para el cálculo de frecuencia global
did_time = 7200  # 2 horas: 7200    ;   4 horas: 14400


def merge_overlapping_events(df, start_col='Event Start Time', dur_col='Duration Increase'):
    """
    Une eventos solapados (IBI < 0). Asume df ORDENADO por start_col.
    Devuelve un nuevo DataFrame con columnas:
      - Event Start Time
      - Duration Increase
      - InterBout Interval
      - Bout
    """
    # Si no hay eventos, devolvemos DF vacío con columnas esperadas
    if df.empty:
        return pd.DataFrame(columns=[start_col, 'Duration Increase', 'InterBout Interval', 'Bout'])

    # Construimos lista de (start, end)
    events = []
    for _, row in df.iterrows():
        start = float(row[start_col])
        dur = float(row[dur_col])
        end = start + dur
        events.append([start, end])

    # Merge de solapados: si start_i < end_prev => IBI < 0 => fusionar
    merged = []
    curr_start, curr_end = events[0]
    for i in range(1, len(events)):
        s, e = events[i]
        if s < curr_end:  # solape: unir
            curr_end = max(curr_end, e)
        else:
            merged.append([curr_start, curr_end])
            curr_start, curr_end = s, e
    merged.append([curr_start, curr_end])

    # Construir DF con merged
    m_df = pd.DataFrame(merged, columns=[start_col, 'Event End Time'])
    m_df['Duration Increase'] = m_df['Event End Time'] - m_df[start_col]

    # Renumerar bouts
    m_df['Bout'] = range(1, len(m_df) + 1)

    # IBI correcto: start_i - end_{i-1}; primera es NaN
    m_df['InterBout Interval'] = m_df[start_col] - m_df['Event End Time'].shift(1)

    # Dejar columnas finales consistentes
    return m_df[[start_col, 'Duration Increase', 'InterBout Interval', 'Bout']]


class App:
    def __init__(self, root):
        self.root = root
        self.setup_ui()
        self.files = []  # rutas de archivos a procesar

    def setup_ui(self):
        self.root.title('Arrastrar y Soltar Archivos')
        self.label = tk.Label(
            self.root,
            text='Arrastra todos los archivos y carpetas aquí',
            bg='lightgrey',
            width=50,
            height=10
        )
        self.label.pack(fill=tk.BOTH, expand=True)

        # Registrar como destino DnD y bindear el drop
        self.label.drop_target_register(DND_FILES)
        self.label.dnd_bind('<<Drop>>', self.drop)

        self.submit_button = ttk.Button(self.root, text='Enviar', command=self.process_files)
        self.submit_button.pack(side=tk.BOTTOM, pady=5)

    def drop(self, event):
        """
        FIX: usar el intérprete Tcl del widget que recibe el evento:
        event.widget.tk.splitlist(event.data)  (NO usar tk.splitlist)
        """
        # 1) Separar correctamente la lista Tcl que viene en event.data
        try:
            items = event.widget.tk.splitlist(event.data)
        except Exception:
            items = [event.data] if event.data else []

        # 2) Normalizar y expandir archivos/carpetas
        nuevos = []
        for item in items:
            path = (item or '').strip()
            # splitlist ya maneja {}, pero por si acaso:
            if path.startswith('{') and path.endswith('}'):
                path = path[1:-1]

            path = os.path.normpath(path)

            if os.path.isdir(path):
                for root_dir, _, files in os.walk(path):
                    for fname in files:
                        if fname.lower().endswith('.csv'):
                            nuevos.append(os.path.join(root_dir, fname))
            else:
                if path.lower().endswith('.csv'):
                    nuevos.append(path)

        # 3) Deduplicar manteniendo previos
        existentes = set(self.files)
        for p in nuevos:
            if p not in existentes:
                self.files.append(p)
                existentes.add(p)

        # 4) Feedback UI
        total = len(self.files)
        if total == 0:
            self.label.config(text="No se detectaron .csv. Arrastra archivos o carpetas nuevamente.")
        else:
            self.label.config(text=f"{total} archivo(s) .csv listo(s) para procesar")

    def process_files(self):
        if not self.files:
            self.label.config(text='No hay archivos pendientes. Arrastra .csv o carpetas.')
            return

        for file in self.files:
            print(f"Procesando archivo: {file}")
            try:
                self.process_and_save_data(file)
            except Exception as e:
                print(f"Error procesando {file}: {e}")

        # Limpiar la lista de archivos después de procesar
        self.files = []
        self.label.config(text='Arrastra todos los archivos y carpetas aquí')

    def calculate_segments(self, data, segments_seconds, label_suffix):
        segments_minutes = [(start / 60, end / 60) for start, end in zip(segments_seconds[:-1], segments_seconds[1:])]
        bouts_sum = []
        time_per_bout_sum = []
        interbout_interval_segments = []
        event_frequency_per_segment = []
        time_segment_labels = []

        for (start_minute, end_minute), end_second in zip(segments_minutes, segments_seconds[1:]):
            # Event Start Time está en segundos, end_second también
            mask = (data['Event Start Time'] >= start_minute * 60) & (data['Event Start Time'] < end_second)
            segment_bouts = data.loc[mask, 'Bout'].count()
            bouts_sum.append(int(segment_bouts))

            time_per_bout_sum.append(float(data.loc[mask, 'Duration Increase'].sum() / 60.0))

            # Sumar IBI dentro del segmento; NaN ignorado
            interbout_interval_sum = float(data.loc[mask, 'InterBout Interval'].sum(skipna=True) / 60.0)
            interbout_interval_segments.append(interbout_interval_sum)

            # Frecuencia = bouts / duración del segmento (min)
            segment_duration_minutes = end_minute - start_minute
            frequency = (segment_bouts / segment_duration_minutes) if segment_duration_minutes > 0 else 0.0
            event_frequency_per_segment.append(float(frequency))

            time_segment_labels.append(f"{int(start_minute)} - {int(end_minute)} {label_suffix}")

        return pd.DataFrame({
            f'Time Segment {label_suffix}': time_segment_labels,
            f'Bouts {label_suffix}': bouts_sum,
            f'Time per Bout {label_suffix}': time_per_bout_sum,
            f'InterBout Interval {label_suffix}': interbout_interval_segments,
            f'Event Frequency per Segment {label_suffix}': event_frequency_per_segment
        })

    def process_and_save_data(self, filepath):
        # Cargar los datos (ISO-8859-1 por tus archivos; si falla, intenta utf-8 como fallback)
        try:
            data = pd.read_csv(filepath, encoding='ISO-8859-1')
        except UnicodeDecodeError:
            data = pd.read_csv(filepath, encoding='utf-8', errors='replace')

        data.columns = data.columns.str.strip()

        # Validaciones mínimas
        for col in ('LeftDuration', 'ElapsedSecs'):
            if col not in data.columns:
                raise ValueError(f"Columna requerida no encontrada: '{col}'")

        # Calcular incremento en 'LeftDuration'
        data['Duration Increase'] = data['LeftDuration'].diff().fillna(data['LeftDuration'])

        # Filtrar los eventos mínimos (tu umbral de 0.5 s)
        data = data[data['Duration Increase'] >= 0.5].copy()

        # Tiempo de inicio del evento
        data['Event Start Time'] = data['ElapsedSecs'] - data['Duration Increase']

        # Ordenar por inicio de evento
        data = data.sort_values('Event Start Time').reset_index(drop=True)

        # FUSIONAR eventos solapados
        merged = merge_overlapping_events(
            data,
            start_col='Event Start Time',
            dur_col='Duration Increase'
        )

        # Calcular totales globales sobre la tabla fusionada
        total_bouts = len(merged)
        total_bout_time = float(merged['Duration Increase'].sum()) if total_bouts > 0 else 0.0

        total_window = did_time  # ventana fija (segundos)
        merged.insert(0, 'Total Bout', pd.NA)
        merged.insert(1, 'Total Bout Time', pd.NA)
        merged.insert(2, 'Event Frequency per minute', pd.NA)

        if total_bouts > 0 and total_window and total_window > 0:
            merged.at[0, 'Total Bout'] = int(total_bouts)
            merged.at[0, 'Total Bout Time'] = float(total_bout_time)
            merged.at[0, 'Event Frequency per minute'] = float(total_bouts / (total_window / 60.0))

        # Definir segmentos
        segments_seconds_10min = list(range(0, total_window + 1, 600))
        segments_seconds_15min = list(range(0, total_window + 1, 900))
        segments_seconds_30min = list(range(0, total_window + 1, 1800))
        segments_seconds_60min = list(range(0, total_window + 1, 3600))

        # Calcular resultados por segmentos sobre la tabla fusionada
        results_10min = self.calculate_segments(merged, segments_seconds_10min, '10min')
        results_15min = self.calculate_segments(merged, segments_seconds_15min, '15min')
        results_30min = self.calculate_segments(merged, segments_seconds_30min, '30min')
        results_60min = self.calculate_segments(merged, segments_seconds_60min, '60min')

        # Concatenar todo
        base_cols = [
            'Event Start Time',
            'Duration Increase',
            'InterBout Interval',
            'Total Bout',
            'Total Bout Time',
            'Event Frequency per minute',
            'Bout'
        ]
        # Asegurar que base_cols existan incluso si merged está vacío
        for col in base_cols:
            if col not in merged.columns:
                merged[col] = pd.Series(dtype='float64')

        final_data = pd.concat(
            [
                merged[base_cols].reset_index(drop=True),
                results_10min, results_15min, results_30min, results_60min
            ],
            axis=1
        )

        # Guardar el DataFrame final (junto al CSV original)
        directory = os.path.dirname(filepath)
        file_name = os.path.splitext(os.path.basename(filepath))[0]
        final_file_path = os.path.join(directory, f"{file_name}_procesado.xlsx")
        final_data.to_excel(final_file_path, index=False)
        print(f"Archivo guardado en: {final_file_path}")


if __name__ == "__main__":
    root = TkinterDnD.Tk()
    app = App(root)
    root.mainloop()
